**Investments: Theory and Data Analysis**, Bates, Boyer, and Fletcher


# Example Chapter 11: Optimal Active Portfolio
In this example we estimate the optimal active tilt towards the zero-cost portfolio that goes long decile 1 short-term reversal portfolio and short decile 10 short-term reversal portfolio from the Ken French Data Library.


### Imports and Setup

In [ ]:
# Import packages
%pip install -q --upgrade farms

import farms as fm
import pandas as pd

# Establish display options when printing DataFrames
pd.set_option('display.max_columns', None)   # Show all columns without truncation
pd.set_option('display.width', 1000)   # Set the display width so output stays on one line
pd.set_option("display.max_rows", 20) # Force truncation if DataFrame has more than 20 rows

### Load in Data Using `farms.load_ken_french_data()`
For full parameter details, see the [`farms` API documentation](https://github.com/boyerb/farms/blob/main/docs/api.md#load_ken_french_data).

`load_ken_french_data()`

**Required Inputs**

- `data_type` — Selects the portfolio granularity. Potential values are `"deciles"` and `"quintiles"`. Example: `data_type='deciles'`.
- `strategy` — Selects the firm characteristic used to sort stocks. Example: `strategy='shorttermreversal'`.

**Optional Inputs**

- `include_factors` — Default is `None`. Adds factor returns to the portfolio DataFrame. Potential values are `None`, `"market"`, `"ff3"`, and `"ff5"`. Example: `include_factors='market'`.
- `frequency` — Default is `"monthly"`. Selects the observation frequency when it is available for the strategy. Potential values are `"daily"`, `"weekly"`, and `"monthly"`. Example: `frequency='monthly'`.
- `details` — Default is `False`. If `True`, returns additional details about the data. Example: `details=True`.

**Outputs**

- `DataFrame` — Portfolio returns indexed by observation date, with decimal return columns and the requested market-factor columns. Example: `df=fm.load_ken_french_data(...)`.

In [ ]:
# Get monthly short-term reversal deciles and market factors
df = fm.load_ken_french_data(
    data_type='deciles',
    strategy='shorttermreversal',
    include_factors='market',
    details=True,
)

# Use the five most recent complete calendar years ending in 2025.
df = df.loc['2021-01':'2025-12'].copy()

# Construct the zero-cost short-term reversal spread: long decile 1,
# short decile 10.
df['dec1_dec10'] = df['dec1'] - df['dec10']
print(df)

### Define Historical Portfolio Inputs
The historical optimizer estimates expected excess returns and the covariance matrix directly from the return history. The first column is the base portfolio and the remaining columns are zero-cost portfolio tilts. Here the market excess return is the base portfolio and the short-term reversal spread is the active factor.

In [ ]:
# The base return must already be an excess return over the risk-free rate.
returns = df[['mkt-rf', 'dec1_dec10']]
returns.head()

### Define the Optimal Active Portfolio
For full parameter details, see the [`farms` portfolio API documentation](https://github.com/boyerb/farms/blob/main/docs/api.md#historical_factor_tilt_portfolio).

`historical_factor_tilt_portfolio()`

This function estimates the mean excess returns and covariance matrix from historical returns, fixes the base exposure at 1.0, and optimizes the zero-cost factor tilt.

**Required Inputs**

- `returns` — DataFrame containing the base excess-return series followed by one or more zero-cost portfolio return series. Example: `returns=returns`.

**Optional Inputs**

- `base_column` — Default is `0`. Identifies the fixed base-return column; DataFrame labels are supported. Example: `base_column='mkt-rf'`.
- `tilt_bounds` — Optional lower and upper bounds for each zero-cost exposure.
- `missing` — Default is `"drop"`. Drops rows with missing or non-finite values; use `"raise"` to reject them.
- `return_result` — Default is `False`. Set to `True` to return optimization diagnostics.

**Outputs**

- `HistoricalFactorTiltResult` — Includes `weights`, `expected_return`, `volatility`, `variance`, `sharpe`, estimated mean excess returns, covariance matrix, observation count, and solver diagnostics.
- `tuple` — With the default `return_result=False`, returns `(weights, expected_return, volatility)`.

In [ ]:
active_result = fm.historical_factor_tilt_portfolio(
    returns,
    base_column='mkt-rf',
    return_result=True,
)

active_exposures = active_result.weights
active_return = active_result.expected_return
active_volatility = active_result.volatility

print('Input columns:', list(returns.columns))
print('Observations used:', active_result.observations)
print('Estimated mean excess returns:', active_result.mean_excess_returns)
print('Optimal exposures:', active_exposures)
print('Expected return:', active_return)
print('Volatility:', active_volatility)
print('Sharpe ratio:', active_result.sharpe)

### Test Trading Strategy

In [ ]:
# For comparison with Ex06-Ken_French.ipynb

# Create realized excess portfolio returns
excess = returns.to_numpy()   # shape (T,2)

# Calculate the excess returns as a 1D array
# @ performs an inner product
excess_array = excess @ active_exposures

# Convert the Series to a DataFrame
excess = pd.DataFrame(excess_array, index=df.index, columns=['excess'])

# compute total return
total_return = excess['excess'] + df['rf']

# Use the same summary function as Ex06 for a transparent comparison.
# With risk_free supplied, summary_stats uses excess-return volatility
# as the Sharpe-ratio denominator.
active_summary = fm.summary_stats(
    returns=total_return.to_frame(name='active_strategy'),
    risk_free=df['rf'],
    frequency='monthly',
)
active_summary

### Try it

Experiment with the notebook and explain how the historical factor-tilt result changes:

- Change the date filter to a different five-year period or compare a shorter and longer sample.
- Change the Ken French `strategy`, such as `momentum`, `size`, or `profitability`, and rebuild the zero-cost spread from the appropriate portfolio columns.
- Compare monthly and daily data when the selected Ken French series supports both frequencies.
- Try `deciles` versus `quintiles` and explain how the portfolio definition changes.
- Add another zero-cost return column to `returns` and see how the optimized exposures and Sharpe ratio change.
- Use `tilt_bounds` to limit the active exposure. How does a constrained solution compare with the unconstrained result?
- Compare the estimated expected return and volatility with the realized strategy summary. What might explain any difference?